# Мультиразметочная классификация рентгеновских изображений

ResNet50 / fastai, 22 метки. Исходный случайный split по снимкам заменён разделением по пациентам.
Нужен проверенный манифест filename,patient_id[,split] плюс исходный train_df.csv.
SOPInstanceUID идентифицирует изображение, его нельзя подставлять вместо пациента.

Исторические F1 доступны в истории Git; новый код не имеет подтверждённых метрик на реальных данных.
Установите requirements-torch.txt. DATA_ROOT содержит train_df.csv и папку train/;
MANIFEST_PATH указывает манифест, OUTPUT_DIR — локальные результаты.

In [ ]:
# Запускайте из корня клонированного репозитория или из папки проекта.
from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'cv_common').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри полного клона My_projects.')
sys.path.insert(0, str(REPO_ROOT))

import os
import json
from functools import partial
import numpy as np
import pandas as pd
from fastai.vision.all import vision_learner, resnet50, F1ScoreMulti, accuracy_multi, set_seed
from fastai.callback.tracker import SaveModelCallback
from sklearn.metrics import classification_report, f1_score, accuracy_score, hamming_loss
from cv_common.data import validate_images, patient_split
from cv_common.data import read_rgb, create_run_directory, save_split_manifest

SEED = 12345
set_seed(SEED, reproducible=True)
DATA_ROOT = Path(os.environ.get('DATA_ROOT', 'data'))
OUTPUT_DIR = create_run_directory(Path(os.environ.get('OUTPUT_DIR', REPO_ROOT / 'artifacts' / 'xray-multilabel')))


In [ ]:
LABELS = ['Abdomen', 'Ankle', 'Cervical_Spine', 'Chest', 'Clavicles', 'Elbow', 'Feet', 'Finger', 'Forearm', 'Hand', 'Hip', 'Knee', 'Lower_Leg', 'Lumbar_Spine', 'Others', 'Pelvis', 'Shoulder', 'Sinus', 'Skull', 'Thigh', 'Thoracic_Spine', 'Wrist']

In [ ]:
raw = pd.read_csv(DATA_ROOT / 'train_df.csv', dtype=str)
df = raw[['SOPInstanceUID', 'Target']].copy()
df['filename'] = df['SOPInstanceUID'].map(lambda x: 'train/' + x + '-c.png')

def decode_target(value):
    if pd.isna(value) or not str(value).strip():
        raise ValueError('Missing multilabel target.')
    tokens = str(value).split()
    if any(t not in {str(i) for i in range(len(LABELS))} for t in tokens):
        raise ValueError('Unknown label identifier.')
    return [LABELS[i] for i in sorted(set(map(int, tokens)))]

df['labels'] = df['Target'].map(decode_target)
manifest = pd.read_csv(os.environ.get('MANIFEST_PATH', DATA_ROOT / 'manifest.csv'), dtype=str)
if 'patient_id' not in manifest:
    raise ValueError('Patient metadata is mandatory.')
cols = ['filename', 'patient_id'] + [column for column in ['split', 'pixel_sha256'] if column in manifest]
df = df.merge(manifest[cols], on='filename', how='left', validate='one_to_one')
parts = patient_split(validate_images(df, DATA_ROOT), SEED)
training_labels = set().union(*parts['train']['labels'])
if training_labels != set(LABELS):
    raise ValueError('Training partition lacks labels. Review patient/class coverage.')
save_split_manifest(parts, DATA_ROOT, OUTPUT_DIR / 'split_manifest.csv')
print({s: len(frame) for s, frame in parts.items()})

In [ ]:
from cv_common.multilabel import build_loaders

development = pd.concat([parts['train'], parts['validation']], ignore_index=True)
development['is_valid'] = development['split'].eq('validation')
dls = build_loaders(development, LABELS)


In [ ]:
# Порог 0.5 фиксируется до финального теста. accuracy_multi — по элементам
# матрицы меток; это не доля снимков с полностью правильным набором меток.
macro_f1 = F1ScoreMulti(thresh=.5, average='macro')
macro_f1.name = 'f1_macro'
micro_f1 = F1ScoreMulti(thresh=.5, average='micro')
micro_f1.name = 'f1_micro'
learn = vision_learner(dls, resnet50, pretrained=True, model_dir=OUTPUT_DIR,
    metrics=[partial(accuracy_multi, thresh=.5),
             macro_f1, micro_f1])
suggestion = learn.lr_find()
learn.fine_tune(50, base_lr=suggestion.valley, cbs=[SaveModelCallback(monitor='valid_loss', fname='best', reset_on_fit=False)])
# SaveModelCallback восстанавливает лучшую модель после fit.
learn.export(OUTPUT_DIR / 'model.pkl')
(OUTPUT_DIR / 'classes.json').write_text(json.dumps(list(dls.vocab)))

## Финальный тест

Не используйте test для выбора эпохи или порога. Если порог подбирается, делайте это
только на validation. Экспорт fastai содержит pickle: загружайте лишь доверенные файлы.

In [ ]:
test = parts['test'].reset_index(drop=True)
test_dl = dls.test_dl(test, with_labels=True, shuffle=False)
probabilities, targets = learn.get_preds(dl=test_dl)
if probabilities.shape != targets.shape or probabilities.shape != (len(test), len(LABELS)):
    raise ValueError('Prediction rows or label vocabulary do not match.')
if not np.isfinite(probabilities.numpy()).all() or not ((probabilities >= 0) & (probabilities <= 1)).all():
    raise ValueError('Invalid probabilities.')
predictions = (probabilities.numpy() >= .5).astype(int)
truth = targets.numpy().astype(int)
report = {
    'threshold': .5,
    'f1_macro': f1_score(truth, predictions, average='macro', zero_division=0),
    'f1_micro': f1_score(truth, predictions, average='micro', zero_division=0),
    'f1_samples': f1_score(truth, predictions, average='samples', zero_division=0),
    'subset_accuracy': accuracy_score(truth, predictions),
    'hamming_loss': hamming_loss(truth, predictions),
    'per_class': classification_report(truth, predictions, target_names=list(dls.vocab), output_dict=True, zero_division=0),
}
(OUTPUT_DIR / 'test_metrics.json').write_text(json.dumps(report, indent=2))
print(pd.DataFrame(report['per_class']).T)

In [ ]:
def classify_image(path, threshold=.5):
    if not 0 <= threshold <= 1:
        raise ValueError('threshold must be between 0 and 1.')
    probabilities = learn.predict(read_rgb(path))[2]
    # Не ограничиваем результат двумя метками: допустимо любое их число, включая 0.
    return {label: float(p) for label, p in zip(dls.vocab, probabilities) if p >= threshold}